> ⚠️ **LEGACY — fixed-τ 1BQF artefact (bannered 2026-07-06).** Any 1BQF (quantum)
> efficiency / false-rate in this notebook is computed at the **fixed τ=0.35 cut**, which
> pins the 1BQF at an artefactual **~70–75 % efficiency** (the cut chops the halved
> outer-true amplitude band — *the cut, not lost physics*). The project headline
> convention is **wp99** (efficiency-first working point; `Toy_Characterisation/WP99_REFRESH_LOG.md`).
> **Canonical replacement: `Larger_Scatter/store_analysis.py` (store-backed; 1BQF at wp99).** Kept as a historical record; not maintained.

# Larger_Scatter — analysis
Three views: scatter-only, inefficiency-only, combined. Group keys: `sigma_scatt`, `hit_ineff`.

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
HERE = Path('.').resolve()
ev = pd.read_csv(HERE/'results'/'ls_events.csv')
print(ev.shape); ev.head()

FileNotFoundError: [Errno 2] No such file or directory: '/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Larger_Scatter/results/ls_events.csv'

In [ ]:
def view(df, fixed_col, fixed_val):
    return df[np.isclose(df[fixed_col], fixed_val)]

scatter_only = view(ev, 'hit_ineff', 0.0)
ineff_only   = view(ev, 'sigma_scatt', 1e-4)
print('scatter-only:', len(scatter_only), '  ineff-only:', len(ineff_only))

In [ ]:
def plot_seg_eff(df, key, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for k, sub in df.groupby(key):
        g = sub.groupby('n_trk').agg(
            mc=('cls_default_segment_efficiency','mean'),
            mq=('q_default_segment_efficiency','mean'),
        ).reset_index()
        axes[0].plot(g['n_trk'], g['mc'], 'o-', label=f'{key}={k:g}')
        axes[1].plot(g['n_trk'], g['mq'], 's-', label=f'{key}={k:g}')
    for ax, t in zip(axes, ['classical seg eff', 'quantum seg eff']):
        ax.set_xscale('log'); ax.set_xlabel('T'); ax.set_title(f'{title} - {t}')
        ax.legend(fontsize=7); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_seg_eff(scatter_only, 'sigma_scatt', 'scatter only')
plot_seg_eff(ineff_only,   'hit_ineff',   'inefficiency only')

In [ ]:
# Combined heatmap at T=200
for T in (200, 1000):
    sub = ev[ev['n_trk']==T]
    if sub.empty: continue
    piv_C = sub.groupby(['sigma_scatt','hit_ineff'])['cls_default_segment_efficiency'].mean().unstack()
    piv_Q = sub.groupby(['sigma_scatt','hit_ineff'])['q_default_segment_efficiency'].mean().unstack()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, P, t in zip(axes, [piv_C, piv_Q], ['classical', 'quantum']):
        im = ax.imshow(P.values, aspect='auto', origin='lower',
                       extent=[P.columns.min(), P.columns.max(),
                               P.index.min(),   P.index.max()], cmap='viridis', vmin=0, vmax=1)
        ax.set_xlabel('hit_ineff'); ax.set_ylabel('sigma_scatt')
        ax.set_title(f'{t} seg eff @ T={T}')
        plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()